In [2]:
import torch
import torch.nn as nn
from torchinfo import summary
from brainvision.constants import *
from brainvision.device import get_device, print_device_info


In [3]:
from brainvision.losses import DiceLoss, FocalLoss, UnifiedFocalLoss
from brainvision.models import Baseline1DDNN, FabeloDNN, HuEtAl1DCNN, LeeEtAl2DCNN, HamidaEtAl3DCNN, HybridSN, SpectralFormer, Fabelo2DCNN, Simple2DCNN

MODEL_REGISTRY = {
    '1D-NN': lambda: Baseline1DDNN(
        input_channels  = N_DECIMATED_BANDS,
        n_classes       = N_CLASSES,
        dropout         = True,
        dropout_rate    = DROPOUT_RATE
    ),

    '1D-NN-Fabelo': lambda: FabeloDNN(
        input_channels = N_DECIMATED_BANDS,
        n_classes      = N_CLASSES,
    ),

    '1D-CNN': lambda: HuEtAl1DCNN(
        input_channels = N_DECIMATED_BANDS,
        n_classes      = N_CLASSES
    ),

    '2D-CNN': lambda: LeeEtAl2DCNN(
        input_channels = N_DECIMATED_BANDS,
        n_classes      = N_CLASSES,
        patch_size     = PATCH_SIZE
    ),
    
    '2D-CNN-Fabelo': lambda: Fabelo2DCNN(
        input_channels = N_DECIMATED_BANDS,
        n_classes      = N_CLASSES,
        patch_size     = PATCH_SIZE
    ),

    '2D-CNN-Simple': lambda: Simple2DCNN(
        input_channels = N_DECIMATED_BANDS,
        n_classes      = N_CLASSES,
        patch_size     = PATCH_SIZE,
    ),

    '3D-CNN': lambda: HamidaEtAl3DCNN(
        input_channels = N_DECIMATED_BANDS,
        n_classes      = N_CLASSES,
        patch_size     = PATCH_SIZE
    ),
    
    'HybridSN': lambda: HybridSN(
        input_channels = N_DECIMATED_BANDS,
        n_classes      = N_CLASSES,
        patch_size     = PATCH_SIZE
    ),
    
    'SpectralFormer': lambda: SpectralFormer(
        input_channels = N_DECIMATED_BANDS,
        n_classes      = N_CLASSES,
        near_band      = SF_NEAR_BAND,
        dim            = SF_DIM,
        depth          = SF_DEPTH,
        heads          = SF_HEADS,
        dim_head       = SF_DIM_HEAD,
        mlp_dim        = SF_MLP_DIM,
        dropout        = SF_DROPOUT,
        emb_dropout    = SF_EMB_DROPOUT,
        mode           = SF_MODE,
    ),
}

PATCH_MODELS = {'2D-CNN', '3D-CNN', 'HybridSN', '2D-CNN-Fabelo', '2D-CNN-Simple'}

LOSS_REGISTRY = {
    'CE'  : lambda w: nn.CrossEntropyLoss(weight=w),
    'FL'  : lambda w: FocalLoss(alpha=w),
    'DL'  : lambda w: DiceLoss(),
    'UFL' : lambda w: UnifiedFocalLoss(alpha=w),
}

In [4]:
def model_summary(model_name: str):
    """
    Print torchinfo summary for a given model.
    Uses the correct input shape based on whether it's
    a pixel model or a patch model.
    """
    model = MODEL_REGISTRY[model_name]()

    if model_name in PATCH_MODELS:
        input_size = (1, N_DECIMATED_BANDS, PATCH_SIZE, PATCH_SIZE)
    else:
        input_size = (1, N_DECIMATED_BANDS)

    print(f"\n{'═'*60}")
    print(f"  {model_name}")
    print(f"{'═'*60}")

    result = summary(
        model,
        input_size   = input_size,
        col_names    = ['num_params', 'kernel_size', 'mult_adds',
                        'input_size', 'output_size'],
        col_width    = 12,
        row_settings = ['var_names'],
        depth        = 4,
        device       = 'cpu',
        verbose      = 0,
    )

    print(result)

In [5]:
device = get_device()
print_device_info()

  Device     : MPS
  Name       : Apple Silicon (MPS)
  Memory     : 19,070 MB total  | N/A free  | N/A used


In [6]:
x_pixel = torch.randn(4, N_DECIMATED_BANDS).to(device)
x_patch = torch.randn(4, N_DECIMATED_BANDS, PATCH_SIZE, PATCH_SIZE).to(device)

print("── Output shape checks ───────────────────────────────")
for name, builder in MODEL_REGISTRY.items():
    m   = builder().to(device)
    x   = x_patch if name in PATCH_MODELS else x_pixel
    out = m(x)
    print(f"  {name:<16} input={tuple(x.shape)}  "
          f"output={tuple(out.shape)}  "
          f"params={sum(p.numel() for p in m.parameters()):,}")

print("\n── Detailed summaries ────────────────────────────────")
for name in MODEL_REGISTRY:
    model_summary(name)

── Output shape checks ───────────────────────────────
  1D-NN            input=(4, 128)  output=(4, 4)  params=17,055,748
  1D-NN-Fabelo     input=(4, 128)  output=(4, 4)  params=4,936
  1D-CNN           input=(4, 128)  output=(4, 4)  params=76,824
  2D-CNN           input=(4, 128, 11, 11)  output=(4, 4)  params=296,580
  2D-CNN-Fabelo    input=(4, 128, 11, 11)  output=(4, 4)  params=142,052
  2D-CNN-Simple    input=(4, 128, 11, 11)  output=(4, 4)  params=19,644
  3D-CNN           input=(4, 128, 11, 11)  output=(4, 4)  params=147,244
  HybridSN         input=(4, 128, 11, 11)  output=(4, 4)  params=4,174,452
  SpectralFormer   input=(4, 128)  output=(4, 4)  params=198,197

── Detailed summaries ────────────────────────────────

════════════════════════════════════════════════════════════
  1D-NN
════════════════════════════════════════════════════════════
Layer (type (var_name))                  Param #      Kernel Shape Mult-Adds    Input Shape  Output Shape
Baseline1DDNN (Baseline1DD